# synthetic-gov-data-kit — Quickstart

Generate structured synthetic US government benefits test cases for LLM reasoning evaluation.

**Python library for generating structured synthetic US government benefits data.**

---

## What you'll do in this notebook

1. Inspect the verified FY2026 SNAP income/benefit limits
2. Generate SNAP eligibility test cases for Virginia (edge-saturated)
3. Inspect a case — scenario, task, expected answer, and rationale trace
4. Export to YAML, JSONL, and CSV
5. Run a batch across multiple states
6. Score a mock model output with the RationaleEvaluator

In [ ]:
# Install if needed
# !pip install synthetic-gov-data-kit

In [1]:
import sys, os
# If running from the repo root:
sys.path.insert(0, os.path.abspath('..'))

from govsynth import Pipeline, BatchPipeline, list_presets
from govsynth.sources.us.snap import SNAPSource
from govsynth.evaluation import RationaleEvaluator

print('govsynth imported successfully')

govsynth imported successfully


## 1. Available Presets

In [2]:
list_presets()


Preset                 Program    Description
----------------------------------------------------------------------
snap.ca                snap       California SNAP FY2026 — BBCE state (no asset test)
snap.md                snap       Maryland SNAP FY2026 — BBCE state
snap.tx                snap       Texas SNAP FY2026 — strict asset test state, non-expansion Medicaid
snap.va                snap       Virginia SNAP FY2026 — strict asset test state
wic.national           wic        WIC FY2026 national income eligibility (185% FPL)



## 2. Inspect Verified FY2026 SNAP Limits

In [3]:
source = SNAPSource(fiscal_year=2026, state='VA')
t = source.thresholds()

print(f'Program: {t.program.upper()}')
print(f'Period: {source.fy_config.period_label} ({source.fy_config.citation_prefix()})')
print(f'FPL basis: {source.fy_config.fpl_year} HHS poverty guidelines')
print(f'Verification: {t.extra["verification_status"].upper()}')
print(f'Source: {t.source}')
print()
print(f'{"HH Size":<10} {"Gross (130% FPL)":<22} {"Net (100% FPL)":<20} {"Max Benefit"}')
print('-' * 65)
for size in range(1, 9):
    lim = t.by_household_size(size)
    print(f'{size:<10} ${lim.gross_monthly:>8,.0f}/month        ${lim.net_monthly:>8,.0f}/month    ${lim.max_benefit:>6,.0f}')
print()
general = f'${t.asset_limit_general:,}' if t.asset_limit_general is not None else 'N/A (BBCE waived)'
elderly = f'${t.asset_limit_elderly_disabled:,}' if t.asset_limit_elderly_disabled is not None else 'N/A'
print(f'Asset limits: {general} general / {elderly} elderly+disabled')
print(f'BBCE state (no asset test): {t.extra["bbce_state"]}')

Program: SNAP
Period: FY2026 (FY2026 (effective October 01, 2025 – September 30, 2026))
FPL basis: 2025 HHS poverty guidelines
Verification: VERIFIED
Source: USDA FNS SNAP FY2026 Cost-of-Living Adjustments Memo (August 13, 2025)

HH Size    Gross (130% FPL)       Net (100% FPL)       Max Benefit
-----------------------------------------------------------------
1          $   1,696/month        $   1,305/month    $   298
2          $   2,292/month        $   1,763/month    $   546
3          $   2,888/month        $   2,221/month    $   785
4          $   3,483/month        $   2,680/month    $   994
5          $   4,079/month        $   3,138/month    $ 1,183
6          $   4,675/month        $   3,596/month    $ 1,421
7          $   5,271/month        $   4,055/month    $ 1,571
8          $   5,867/month        $   4,513/month    $ 1,789

Asset limits: N/A (BBCE waived) general / $4,500.0 elderly+disabled
BBCE state (no asset test): True


## 3. Generate 20 SNAP Cases (Virginia, Edge-Saturated)

In [4]:
pipeline = Pipeline.from_preset('snap.va')
cases = pipeline.generate(n=20, seed=42)

print(f'\nGenerated {len(cases)} cases')
print(f'Eligible:   {sum(1 for c in cases if c.expected_outcome == "eligible")}')
print(f'Ineligible: {sum(1 for c in cases if c.expected_outcome == "ineligible")}')

# Difficulty breakdown
from collections import Counter
diff_counts = Counter(c.difficulty.value for c in cases)
print(f'Difficulty: {dict(diff_counts)}')

⠋ Generating 20 unknown.va cases... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00

✓ Generated 20 valid cases.


Generated 20 cases
Eligible:   8
Ineligible: 12
Difficulty: {'hard': 15, 'medium': 5}


## 4. Inspect a Single Case

In [6]:
# Find a 'hard' eligible case to inspect
hard_cases = [c for c in cases if c.difficulty.value == 'hard']
case = hard_cases[0] if hard_cases else cases[0]

print('=== CIVBENCH ID ===')
print(case.case_id)
print()
print('=== SCENARIO ===')
print(case.scenario.summary)
print(f'  HH size: {case.scenario.household_size}, Gross: ${case.scenario.monthly_gross_income:,.2f}, '
      f'Net: ${case.scenario.monthly_net_income:,.2f}, Assets: ${case.scenario.liquid_assets:,.2f}')
print()
print('=== TASK ===')
print(case.task.instruction[:200] + '...')
print()
print('=== EXPECTED OUTCOME ===')
print(case.expected_outcome.upper())
print()
print('=== EXPECTED ANSWER ===')
print(case.expected_answer)
print()
print('=== RATIONALE TRACE ===')
print(case.rationale_trace.to_plain_text())
print()
print('=== VARIATION TAGS ===')
print(case.variation_tags)

=== CIVBENCH ID ===
snap.va.eligibility.net_income_limit.above_limit.ineligible.hh1.38b41c

=== SCENARIO ===
Jonathan Campbell is a 56-year-old individual in Danaland, VA. Their household has $1,909/month gross income and $400 in savings.
  HH size: 1, Gross: $1,908.81, Net: $1,318.05, Assets: $400.00

=== TASK ===
Based on the household's situation described above, determine whether this household is eligible for SNAP (Supplemental Nutrition Assistance Program) benefits. Show your reasoning step by step, citing...

=== EXPECTED OUTCOME ===
INELIGIBLE

=== EXPECTED ANSWER ===
This household is INELIGIBLE for SNAP benefits (FY2026).

Reason: Ineligible: gross income $1,908.81 exceeds $1,696.00 (130% FPL, FY2026, 1-person HH)

Applicable limits for a 1-person household: Gross $1,696.00/month (130% FPL), Net $1,305.00/month (100% FPL), Assets N/A (BBCE waived) (general).

=== RATIONALE TRACE ===
Step 1: Check gross income limit
  Rule: 7 CFR 273.9(a)(1)
  Computation: $1,908.81 > $1,696.0

## 5. Export to Multiple Formats

In [7]:
import os
os.makedirs('./output', exist_ok=True)

# YAML (one file per case)
pipeline.save(cases, './output/snap_va/', formats=['yaml'])

# JSONL for fine-tuning
pipeline.save(cases, './output/snap_va.jsonl', formats=['jsonl'])

# CSV for review
pipeline.save(cases, './output/snap_va.csv', formats=['csv'])

print('\nOutput files:')
for f in os.listdir('./output'):
    print(f'  {f}')

✓ Saved YAML → output/snap_va/

✓ Saved JSONL → output/snap_va.jsonl

✓ Saved CSV → output/snap_va.csv


Output files:
  snap_va.jsonl
  snap_va
  snap_va.csv


## 6. Generate Across Multiple States (Batch)

In [8]:
batch = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
all_cases = batch.generate(n_per_pipeline=15, seed=42)

# Show jurisdiction breakdown
from collections import Counter
by_state = Counter(c.jurisdiction for c in all_cases)
print('Cases by jurisdiction:')
for j, n in sorted(by_state.items()):
    print(f'  {j}: {n}')

# Compare eligibility rates — CA (BBCE) should differ from TX (strict asset test)
print()
for state in ['us.va', 'us.ca', 'us.tx']:
    state_cases = [c for c in all_cases if c.jurisdiction == state]
    if state_cases:
        elig_rate = sum(1 for c in state_cases if c.expected_outcome == 'eligible') / len(state_cases)
        bbce = any('bbce_state' in c.variation_tags for c in state_cases)
        print(f'  {state}: {elig_rate:.0%} eligible, BBCE={bbce}')

⠋ Generating 15 unknown.tx cases... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00

✓ Generated 15 valid cases.

Batch complete: 45 total cases from 3 pipelines.

Cases by jurisdiction:
  us.ca: 15
  us.tx: 15
  us.va: 15

  us.va: 33% eligible, BBCE=True
  us.ca: 47% eligible, BBCE=True
  us.tx: 53% eligible, BBCE=False


## 7. Score a Mock Model Output with RationaleEvaluator

In [11]:
evaluator = RationaleEvaluator()

# Use the hard case from section 4
test_case = hard_cases[0] if hard_cases else cases[0]

good_output = (
    f"The household has {test_case.scenario.household_size} members. "
    f"Gross income ${test_case.scenario.monthly_gross_income:,.2f} must be checked against "
    f"the 130% FPL limit per 7 CFR 273.9(a)(1). After the 20% earned income deduction "
    f"and standard deduction under 7 CFR 273.9(c)(2), net income is "
    f"${test_case.scenario.monthly_net_income or 0:,.2f}. "
    f"The asset limit of $3,000 applies per 7 CFR 273.8(b). "
    f"This household is {'eligible' if test_case.expected_outcome == 'eligible' else 'ineligible'} for SNAP."
)

score = evaluator.score(test_case, good_output)
print(score)
print(f'\nSteps covered: {score.steps_covered}')
print(f'Steps missed:  {score.steps_missed}')
print(f'Rules cited:   {score.rules_cited}')

RationaleScore(snap.va.eligibility.net_income_limit.above_limit.ineligible.hh1.38b41c) [PASS]
  overall=0.82  step_coverage=1.00  rule_accuracy=1.00  conclusion=0.50
  predicted_outcome='ambiguous'  expected='ineligible'

Steps covered: ['Check gross income limit', 'Eligibility determination']
Steps missed:  []
Rules cited:   ['7 CFR 273.9(a)(1)']


## 8. At-Threshold Profile Factory

Generate profiles precisely at policy boundaries — the edge cases where models fail.

In [12]:
from govsynth.profiles import USHouseholdProfile

# Generate a household exactly at the 3-person SNAP gross income limit
at_limit = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=0.0,  # exactly at limit
    seed=42,
)
print(f'AT LIMIT:    gross=${at_limit.monthly_gross_income:,.2f} (should be $2,888.00)')

# Just above the limit — should be ineligible
above = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=0.01,  # 1% above
    seed=42,
)
print(f'ABOVE LIMIT: gross=${above.monthly_gross_income:,.2f} (should be ~$2,916.88)')

# Just below the limit — should be eligible
below = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=-0.01,  # 1% below
    seed=42,
)
print(f'BELOW LIMIT: gross=${below.monthly_gross_income:,.2f} (should be ~$2,859.12)')

# Verify eligibility for each
source = SNAPSource(fiscal_year=2026, state='VA')
for label, profile in [('AT LIMIT', at_limit), ('ABOVE', above), ('BELOW', below)]:
    eligible, reason = source.is_eligible(
        household_size=profile.household_size,
        gross_income=profile.monthly_gross_income,
        net_income=profile.monthly_net_income,
        liquid_assets=profile.liquid_assets,
    )
    print(f'{label:12} → {"ELIGIBLE" if eligible else "INELIGIBLE"}: {reason[:80]}')

AT LIMIT:    gross=$2,888.00 (should be $2,888.00)
ABOVE LIMIT: gross=$2,916.88 (should be ~$2,916.88)
BELOW LIMIT: gross=$2,859.12 (should be ~$2,859.12)
AT LIMIT     → ELIGIBLE: Eligible: all tests passed (gross income, net income, assets)
ABOVE        → INELIGIBLE: Ineligible: gross income $2,916.88 exceeds $2,888.00 (130% FPL, FY2026, 3-person
BELOW        → ELIGIBLE: Eligible: all tests passed (gross income, net income, assets)


## Next Steps

- See `notebooks/02_snap_edge_cases.ipynb` for deeper SNAP edge case analysis
- See `notebooks/03_wic_eligibility.ipynb` for WIC case generation
- See `notebooks/04_batch_pipeline.ipynb` to generate a batch test suite
- Run `python scripts/verify_thresholds.py` to check threshold verification status
- See [CLAUDE.md](../CLAUDE.md) for AI coding assistant context
- See [AGENTS.md](../AGENTS.md) for agentic workflow playbooks